# Customer Churn Prediction with a Neural Network

This beginner-friendly notebook predicts whether a telecom customer is likely to **churn** (leave the company). It also demonstrates the standard machine-learning workflow: understand data → prepare it → train → evaluate.

## 1. What are we predicting?

`Churn` is our **target**: `Yes` means a customer left, and `No` means they stayed. All other columns are **features**—information used to make the prediction.

A model does not know business rules in advance. It looks at examples from past customers and learns patterns associated with churn.

In [ ]:
# Run this cell first. If TensorFlow is missing, install it with: pip install tensorflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

import tensorflow as tf
from tensorflow import keras

# Makes results more repeatable when you run the notebook again.
np.random.seed(42)
tf.random.set_seed(42)

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv('customer_churn.csv')

print(f'Dataset shape: {df.shape[0]} rows and {df.shape[1]} columns')
df.head()

In [ ]:
# Check the class balance. Churn data is usually imbalanced: more customers stay than leave.
df['Churn'].value_counts().rename_axis('Churn').to_frame('Customers')

## 3. Clean the data

`customerID` is unique for every customer, so it cannot help the model learn a general pattern. `TotalCharges` should be a number, but some rows have blank values. We convert blanks to missing values and remove only those few incomplete rows.

In [ ]:
df = df.drop(columns='customerID').copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna().copy()

print(f'Rows after cleaning: {len(df)}')
df.info()

## 4. Explore two useful features

These charts do not prove causation, but they help us see how churn differs between groups.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', ax=axes[0])
axes[0].set_title('Churn by tenure')
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple='stack', ax=axes[1])
axes[1].set_title('Churn by monthly charges')
plt.tight_layout()

## 5. Separate inputs (`X`) and answer (`y`)

Machine-learning code conventionally calls the input table `X` and the answer column `y`. We change `Yes`/`No` into `1`/`0` because the neural network works with numbers.

In [ ]:
X = df.drop(columns='Churn')
y = df['Churn'].map({'No': 0, 'Yes': 1})

# Keep the same churn proportion in both datasets. The test set is never used for training.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Training examples:', len(X_train))
print('Testing examples:', len(X_test))

## 6. Convert features into model-ready numbers

Neural networks require numeric input. We scale numeric values to 0–1, and **one-hot encode** text categories. For example, the `Contract` column becomes separate columns such as `Contract_Month-to-month`.

Important: the transformer is *fit only on training data*. This prevents information from the test set leaking into training.

In [ ]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer([
    ('numeric', MinMaxScaler(), numeric_features),
    ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print('Features after encoding:', X_train_ready.shape[1])
print('Training matrix shape:', X_train_ready.shape)

## 7. Build the ANN

A **Dense** layer connects every input to every neuron in the next layer. The hidden layers use ReLU to learn patterns. The final sigmoid neuron returns a probability between 0 and 1—for example, `0.78` means an estimated 78% chance of churn.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(X_train_ready.shape[1],)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 8. Train the model

`validation_split` holds back part of the training data to monitor learning. Early stopping stops training when validation loss no longer improves, helping reduce overfitting.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train_ready, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy loss')
plt.title('Learning curve')
plt.legend()
plt.show()

## 9. Evaluate on unseen test data

The test set simulates new customers. A probability of 0.5 or higher is classified as churn here. In a real business, you might choose a lower threshold if catching more potential churners is more valuable.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_ready, y_test, verbose=0)
churn_probability = model.predict(X_test_ready, verbose=0).ravel()
y_pred = (churn_probability >= 0.5).astype(int)

print(f'Test accuracy: {test_accuracy:.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['Stayed (0)', 'Churned (1)']))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Stayed', 'Churned'], cmap='Blues'
)
plt.title('Confusion matrix on the test set')
plt.show()

## 10. How to read the metrics

- **Accuracy**: percentage of all predictions that were correct. It can be misleading when most customers do not churn.
- **Precision for Churned (1)**: when the model flags a customer as likely to churn, how often is it right?
- **Recall for Churned (1)**: of all customers who really churned, how many did the model find?
- **F1-score**: a single score that balances churn precision and recall.

For retention campaigns, recall is often important: missing a likely churner may mean losing that customer.

## Next experiments

1. Change the probability threshold from `0.5` to `0.4` and compare churn recall and precision.
2. Try different layer sizes or dropout.
3. Compare this ANN with a simpler model such as logistic regression.
4. Save the trained model and preprocessing transformer before using it with new customer data.